# NB02 — completed-run audit and direct CSV recovery

**CPU only. No training, model changes, test-based reselection or ZIP upload.**

**Repair r2: fresh-session output directories are created before retrieval.**
Re-import this file and Run All. Existing complete downloads in the same session
are reused; no training notebook or completed checkpoint needs rerunning.

The r1 run is already complete. This notebook reads its immutable HF evidence,
checks every checkpoint/archive member, independently recalculates saved scores,
recovers detailed CSV and publishes those CSV files directly in one additive commit.

1. Accelerator **None / CPU**; Internet ON.
2. Attach **shanmuk4622/tire-dataset-prepared-phase2** from Kaggle.
3. Enable secret **HF_TOKEN**, write access to **Shanmuk4622/tyre-wear-study**.
4. Run All in order. The final publication cell adds a `posthoc-csv/2026-10-28/`
   folder under the original protocol; it does not overwrite model/checkpoint files.

It downloads about **390 MiB** of legacy evidence once. Local ZIPs are read only
to recover the historical format; **no ZIP/NPZ/checkpoint is uploaded by this notebook**.
Native input masks stay read-only. About 3 GiB free working space is required.
Existing Kaggle PyTorch/Hub dependencies are retained; no global package downgrade.

Source revision: `a0296f18e02b12d28f7e51377783d68538032209`.
Scores remain low/high-only, two-test-tyre internal results. Added metrics are
explicitly post-hoc derivations, not previously unlogged observations.


In [1]:
import base64, gzip, hashlib, json, os, shutil, subprocess, sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient
import torch, numpy, PIL, requests, huggingface_hub
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
WORK = Path('/kaggle/working/treadone_csv_recovery')
WORK.mkdir(parents=True, exist_ok=True)
assert shutil.disk_usage(WORK).free > 3*1024**3, 'Need 3 GiB working free for recovery/verification.'
scripts = WORK/'analysis'
scripts.mkdir(parents=True, exist_ok=True)
(WORK/'completion_20261028').mkdir(parents=True, exist_ok=True)
payload = json.loads(gzip.decompress(base64.b64decode('...')))
for name, value in payload.items():
    assert Path(name).name == name
    (scripts/name).write_bytes(base64.b64decode(value))
print('CPU recovery sources extracted. HF_TOKEN loaded without displaying it.')
def launch(script, *arguments):
    subprocess.run([sys.executable, '-B', '-u', str(scripts/script), *arguments], check=True)


CPU recovery sources extracted. HF_TOKEN loaded without displaying it.


## 1. Retrieve and verify the historical evidence
A restart reuses completed local downloads; immutable source hashes remain fixed.

In [2]:
launch('fetch_completed_results.py', '--bundles')

Fetched INDEX.json 1043 bytes
Fetched LEASE.json 26 bytes
Fetched CONTRACT.json 3765 bytes
Fetched OVERLAY.json 806991 bytes
Fetched ENVIRONMENT.json 466 bytes
Fetched DATA_PREFLIGHT.json 1785 bytes
Fetched report/REPORT.json 790504 bytes
Fetched report/EXPORT.json 4279 bytes
Fetched report/EVALUATION_seed1.json 262007 bytes
Fetched report/EVALUATION_seed2.json 262684 bytes
Fetched report/EVALUATION_seed3.json 263756 bytes
runs/seed1/latest.zip 32 MiB downloaded
runs/seed1/latest.zip 64 MiB downloaded
runs/seed1/latest.zip 96 MiB downloaded
runs/seed1/latest.zip 128 MiB downloaded
SHA256 verified bundle runs/seed1/latest.zip
runs/seed2/latest.zip 32 MiB downloaded
runs/seed2/latest.zip 64 MiB downloaded
runs/seed2/latest.zip 96 MiB downloaded
runs/seed2/latest.zip 128 MiB downloaded
SHA256 verified bundle runs/seed2/latest.zip
runs/seed3/latest.zip 32 MiB downloaded
runs/seed3/latest.zip 64 MiB downloaded
runs/seed3/latest.zip 96 MiB downloaded
SHA256 verified bundle runs/seed3/latest.

## 2. Audit and write CSV
This checks 8,820 updates, 180 validation epochs, raw test predictions and source-mask hashes; it performs no optimizer step.

In [3]:
launch('audit_and_export_csv.py')
launch('publish_recovered_csv.py', '--prepare-only')
print((WORK/'completion_20261028/CSV_README.md').read_text())

Seed 1 checkpoint, 60 validation epochs, both raw test endpoints and scalar histories verified
Seed 2 checkpoint, 60 validation epochs, both raw test endpoints and scalar histories verified
Seed 3 checkpoint, 60 validation epochs, both raw test endpoints and scalar histories verified
{
  "status": "passed",
  "rows": {
    "training_steps.csv": 8820,
    "training_samples.csv": 70560,
    "validation_epochs.csv": 180,
    "validation_predictions.csv": 14580,
    "validation_confusion.csv": 1620,
    "class_metrics.csv": 558,
    "calibration_bins.csv": 674,
    "point_predictions.csv": 91188,
    "review_flags.csv": 1813,
    "test_predictions.csv": 618,
    "test_region_stats.csv": 1236,
    "test_group_metrics.csv": 12,
    "run_summary.csv": 6,
    "sample_exposures.csv": 33,
    "gpu_resume_checks.csv": 3,
    "completion_checks.csv": 3,
    "numerical_events.csv": 0
  },
  "summaries": [
    {
      "seed": 1,
      "endpoint": "selected",
      "selected_epoch": 23,
      "valida

## 3. Publish directly readable CSV
One batched commit, verified file hashes; no ZIP upload and no modification of the original training namespace files. Retry the same cell after a transient publication-only failure.

In [4]:
launch('publish_recovered_csv.py')
print((WORK/'completion_20261028/CSV_PUBLICATION.json').read_text())

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...csv/point_predictions.csv: 100%|█████████▉| 13.5MB / 13.5MB            

Processing Files (0 / 1)      : 100%|█████████▉| 13.5MB / 13.5MB,   ???B/s  
New Data Upload               : 100%|█████████▉| 13.5MB / 13.5MB,   ???B/s  

Processing Files (1 / 1)      : 100%|██████████| 13.5MB / 13.5MB,   ???B/s  
New Data Upload               : 100%|██████████| 13.5MB / 13.5MB,   ???B/s  

Processing Files (1 / 1)      : 100%|██████████| 13.5MB / 13.5MB, 1.32MB/s  
New Data Upload               : 100%|██████████| 13.5MB / 13.5MB, 1.32MB/s  
  ...csv/point_predictions.csv: 100%|██████████| 13.5MB / 13.5MB            


{"status": "published_and_verified", "revision": "91ccd3315bd296a76a8b583a6d0f2515ccb3d370", "destination": "phase3/tread-one-r1/371c40c0c6aaedaf5c5970f7721a284b042494db7bb330065f6cbde8fc3a0b8c/posthoc-csv/2026-10-28", "source_revision": "a0296f18e02b12d28f7e51377783d68538032209", "files": 25, "zip_files_uploaded": 0}
{
  "status": "published_and_verified",
  "revision": "91ccd3315bd296a76a8b583a6d0f2515ccb3d370",
  "destination": "phase3/tread-one-r1/371c40c0c6aaedaf5c5970f7721a284b042494db7bb330065f6cbde8fc3a0b8c/posthoc-csv/2026-10-28",
  "source_revision": "a0296f18e02b12d28f7e51377783d68538032209",
  "files": 25,
  "zip_files_uploaded": 0
}



## 4. Inspect local results
The same individual CSV files remain in Kaggle Output. The dataset viewer can browse the new HF folder directly.

In [5]:
import pandas as pd
display(pd.read_csv(WORK/'completion_20261028/csv/run_summary.csv'))
print('CSV folder:', WORK/'completion_20261028/csv')

,seed,endpoint,selected_epoch,validation_utility,images,balanced_accuracy_observed,accuracy,macro_f1_three_classes_zero_for_missing,ordinal_mae,quadratic_weighted_kappa,...,low_recall,mid_recall,high_recall,mean_two_region_dice,mean_two_region_iou,mask_derived_box_iou,human_point_mae_width_fraction,human_point_images,evidence_logit_max_residual,joint_validation_utility
0,1,selected,23,0.737886,103,0.500000,0.398058,0.262821,0.815534,0.332253,...,0.0,NaN,1.000000,0.978524,0.958152,0.892976,0.010671,25,0.0,0.710471
1,1,fixed_final,23,0.737886,103,0.500000,0.398058,0.329317,0.611650,0.599078,...,0.0,NaN,1.000000,0.975792,0.953014,0.872294,0.010763,25,0.0,0.709238
2,2,selected,30,0.850679,103,0.024390,0.019417,0.031008,0.980583,0.045509,...,0.0,NaN,0.048780,0.976618,0.954551,0.880984,0.013050,25,0.0,0.464862
3,2,fixed_final,30,0.850679,103,0.219512,0.174757,0.203390,0.825243,0.337696,...,0.0,NaN,0.439024,0.977633,0.956463,0.853176,0.013914,25,0.0,0.560185
4,3,selected,20,0.972213,103,0.500000,0.398058,0.333333,0.601942,0.614232,...,0.0,NaN,1.000000,0.984187,0.968905,0.971183,0.011960,25,0.0,0.708584
5,3,fixed_final,20,0.972213,103,0.036585,0.029126,0.045455,0.970874,0.067367,...,0.0,NaN,0.073171,0.981454,0.963681,0.928607,0.014185,25,0.0,0.469246


CSV folder: /kaggle/working/treadone_csv_recovery/completion_20261028/csv
